In [ ]:
import json
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

In [ ]:
with h5py.File('./simulation.h5', 'r') as f:
    config = json.loads(f.attrs['config'])
    cols   = list(f['metrics/simulation_metrics'].attrs['columns'])
    arr    = f['metrics/simulation_metrics'][:]
    phylo_arr = f['phylogeny'][:] if 'phylogeny' in f else np.empty((0, 3), dtype='int32')

data  = pd.DataFrame(arr, columns=cols)
phylo = pd.DataFrame(phylo_arr, columns=['parent_species', 'child_species', 'step'])

In [ ]:
final   = data.iloc[-1]
initial = data.iloc[0]

gene_cols  = [c for c in data.columns if c.startswith('Average ') and c != 'Average Age']
death_cols = [c for c in data.columns if c.startswith('Deaths from ')]

pd.DataFrame({
    'Deaths (final cumulative)': {
        'Total': int(final['Cumulative Deaths']),
        **{col.replace('Deaths from ', ''): int(final[col]) for col in death_cols},
    },
    'Population': {
        'Final': int(final['Population Count']),
        'Peak':  int(data['Population Count'].max()),
        'Min':   int(data['Population Count'].min()),
    },
    'Species': {
        'Final': int(final['Number of Species']),
        'Peak':  int(data['Number of Species'].max()),
    },
    'Gene drift (initial -> final)': {
        col.replace('Average ', ''): f"{initial[col]:.2f} -> {final[col]:.2f}"
        for col in gene_cols
    },
}).fillna('')

In [ ]:
plt.figure(figsize=(12, 6))
# Average Age is a runtime property, not a gene — shown dashed for reference
if config['enable_aging']:
    plt.plot(data['Timestep'], data['Average Age'], label='Average Age', linestyle='--', alpha=0.6)
# gene_cols is auto-discovered from HDF5 — always in sync with the GENES registry
for col in gene_cols:
    plt.plot(data['Timestep'], data[col], label=col.replace('Average ', ''))
plt.xlabel('Timestep')
plt.ylabel('Value')
plt.title('Time Series of Average Gene Values')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
# Auto-discovered from HDF5 columns — always in sync with METRIC_COLUMNS
death_cols = [c for c in data.columns if c.startswith('Deaths from ')]
for col in death_cols:
    plt.plot(data['Timestep'], data[col], label=col)
plt.xlabel('Timestep')
plt.ylabel('Cumulative deaths')
plt.title('Agent Deaths Over Time')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(data['Timestep'], data['Number of Species'], label='Number of Species')
plt.xlabel('Timestep')
plt.ylabel('Number of Species')
plt.title('Number of Species Over Time')
plt.legend()
plt.show()

In [ ]:
import networkx as nx

if phylo.empty:
    print('No speciation events recorded.')
else:
    G = nx.DiGraph()
    G.add_node(1, step=0)
    for _, row in phylo.iterrows():
        child, parent, step = int(row['child_species']), int(row['parent_species']), int(row['step'])
        G.add_node(child, step=step)
        G.add_edge(parent, child)

    y_pos    = {}
    leaf_idx = [0]
    for node in nx.dfs_postorder_nodes(G, 1):
        children = list(G.successors(node))
        if not children:
            y_pos[node] = leaf_idx[0]
            leaf_idx[0] += 1
        else:
            y_pos[node] = sum(y_pos[c] for c in children) / len(children)

    x_pos    = {n: G.nodes[n]['step'] for n in G.nodes}
    n_leaves = leaf_idx[0]

    fig, ax = plt.subplots(figsize=(14, max(6, n_leaves * 0.25)))
    for parent, child in G.edges():
        px, py = x_pos[parent], y_pos[parent]
        cx, cy = x_pos[child],  y_pos[child]
        ax.plot([px, cx], [py, py], color='steelblue', lw=0.6, alpha=0.6)
        ax.plot([cx, cx], [py, cy], color='steelblue', lw=0.6, alpha=0.6)
    xs = [x_pos[n] for n in G.nodes]
    ys = [y_pos[n] for n in G.nodes]
    ax.scatter(xs, ys, s=8, color='steelblue', zorder=3)
    ax.set_xlabel('Timestep')
    ax.set_yticks([])
    ax.set_title(f'Phylogenetic Tree — {G.number_of_nodes()} species, depth {nx.dag_longest_path_length(G)}')
    plt.tight_layout()
    plt.show()

    print(f'Total speciation events : {len(phylo)}')
    print(f'Max clade depth         : {nx.dag_longest_path_length(G)}')
    most_prolific = max(G.nodes, key=lambda n: len(nx.descendants(G, n)))
    print(f'Most prolific ancestor   : species {most_prolific} ({len(nx.descendants(G, most_prolific))} descendants)')
